# Email Spam Classification from Scratch

## Part 1: Text Data Preprocessing and Bag of Words

In this section, we preprocess the raw text data to prepare it for our machine learning models. We convert the text into numerical vectors using the Bag of Words (BoW) technique, map the string labels to numeric values, clean the text, and extract the top 15 most frequent words.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

dataframe = pd.read_csv("emails_1.csv")

vectorizer = CountVectorizer(max_features=15, lowercase=True, token_pattern=r'[a-zA-Z]+')
bag_of_words = vectorizer.fit_transform(dataframe['text'])

bag_of_words_df = pd.DataFrame(bag_of_words.toarray(), columns=vectorizer.get_feature_names_out())
bag_of_words_df['status'] = dataframe['status'].map({"spam": 1, "ham": 0}).values

display(bag_of_words_df)

,a,and,by,claim,enjoy,for,free,get,our,please,project,the,to,you,your,status
0,1,0,0,1,0,0,2,0,1,0,0,0,1,1,1,1
1,1,1,0,0,0,0,0,0,0,1,2,1,0,0,1,0
2,1,0,0,0,1,2,2,1,1,0,0,0,0,1,0,1
3,0,1,1,0,0,0,0,0,0,1,0,1,0,0,1,0
4,1,0,0,1,1,1,1,0,0,0,0,0,0,1,2,1
5,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,1
6,0,0,0,0,0,1,0,0,0,1,1,1,0,0,0,0
7,0,0,0,1,0,0,1,0,0,0,0,1,1,0,2,1
8,0,1,1,0,0,1,0,1,0,0,0,1,1,1,1,0
9,0,0,0,0,0,1,0,0,0,0,0,0,1,0,2,1


### Loading the Prepared Dataset
Here, we load `emails_2.csv`, which is a larger dataset containing 5172 real emails already preprocessed into a Bag of Words format with 3000 features. We will use this dataset for training our models.

In [2]:
dataframe_2 = pd.read_csv("emails_2.csv")

print("\033[1mRaw Data:\033[0m")
display(dataframe_2.head(10))

Raw Data:


,Email No.,the,to,ect,and,for,of,a,you,hou,...,connevey,jay,valued,lay,infrastructure,military,allowing,ff,dry,Prediction
0,Email 1,0,0,1,0,0,0,2,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Email 2,8,13,24,6,6,2,102,1,27,...,0,0,0,0,0,0,0,1,0,0
2,Email 3,0,0,1,0,0,0,8,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Email 4,0,5,22,0,5,1,51,2,10,...,0,0,0,0,0,0,0,0,0,0
4,Email 5,7,6,17,1,5,2,57,0,9,...,0,0,0,0,0,0,0,1,0,0
5,Email 6,4,5,1,4,2,3,45,1,0,...,0,0,0,0,0,0,0,0,0,1
6,Email 7,5,3,1,3,2,1,37,0,0,...,0,0,0,0,0,0,0,0,0,0
7,Email 8,0,2,2,3,1,2,21,6,0,...,0,0,0,0,0,0,0,1,0,1
8,Email 9,2,2,3,0,0,1,18,0,0,...,0,0,0,0,0,0,0,0,0,0
9,Email 10,4,4,35,0,1,0,49,1,16,...,0,0,0,0,0,0,0,0,0,0


## Part 2: Train-Test Split

This section implements a function to split the dataset into training (80%) and testing (20%) sets to ensure we can properly evaluate the generalization of our models.

In [3]:
def train_test_split(X, y, test_size=0.2, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)

    indices = np.random.permutation(len(X))
    test_size = int(len(X) * test_size)
    test_indices = indices[:test_size]
    train_indices = indices[test_size:]

    X_train = X[train_indices]
    X_test = X[test_indices]
    y_train = y[train_indices]
    y_test = y[test_indices]

    return X_train, X_test, y_train, y_test

X = dataframe_2.drop(['Email No.', 'Prediction'], axis=1).values
y = dataframe_2['Prediction'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\033[1mNumber of samples in X:\033[0m", len(X))
print("\033[1mNumber of samples in X_train:\033[0m", len(X_train))
print("\033[1mNumber of samples in X_test:\033[0m", len(X_test))


Number of samples in X: 5172
Number of samples in X_train: 4138
Number of samples in X_test: 1034


## Part 3: Implementation of Evaluation Metrics

Before training the models, we implement custom functions to calculate key evaluation metrics for binary classification from scratch: Accuracy, Precision, Recall, and F1-Score.

In [4]:
def accuracy(y_true, y_pred):
    correct_predictions = np.sum(y_true == y_pred)
    total_samples = len(y_true)
    return correct_predictions / total_samples

def precision(y_true, y_pred):
    true_positives = np.sum((y_true == 1) & (y_pred == 1))
    predicted_positives = np.sum(y_pred == 1)
    return true_positives / predicted_positives if predicted_positives > 0 else 0

def recall(y_true, y_pred):
    true_positives = np.sum((y_true == 1) & (y_pred == 1))
    actual_positives = np.sum(y_true == 1)
    return true_positives / actual_positives if actual_positives > 0 else 0

def f1_score(y_true, y_pred):
    prec = precision(y_true, y_pred)
    rec = recall(y_true, y_pred)
    return 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0

y_true = np.array([0, 1, 1, 0, 1])
y_pred = np.array([0, 1, 0, 0, 1])

print("\033[1mExample Metric Results:\033[0m")
print(f"Accuracy: {accuracy(y_true, y_pred)*100:.2f}%")
print(f"Precision: {precision(y_true, y_pred)*100:.2f}%")
print(f"Recall: {recall(y_true, y_pred)*100:.2f}%")
print(f"F1 Score: {f1_score(y_true, y_pred)*100:.2f}%")


Example Metric Results:
Accuracy: 80.00%
Precision: 100.00%
Recall: 66.67%
F1 Score: 80.00%


## Part 4: Logistic Regression Implementation

In this section, we implement a custom Logistic Regression model from scratch. The data is normalized using a custom `StandardScaler` prior to training to ensure better convergence. The model is then trained and its performance is evaluated on the test set.

In [5]:
class StandardScaler:
    def __init__(self):
        self.mean_ = None
        self.scale_ = None
        self.fitted_ = False

    def fit(self, X):
        X = np.array(X, dtype=float)
        self.mean_ = X.mean(axis=0)
        self.scale_ = X.std(axis=0, ddof=0)
        self.scale_[self.scale_ == 0] = 1.0
        self.fitted_ = True

        return self

    def transform(self, X):
        if not self.fitted_:
            raise ValueError("First you need to call fit.")

        X = np.array(X, dtype=float)
        return (X - self.mean_) / self.scale_

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, X_scaled):
        if not self.fitted_:
            raise ValueError("First you need to call fit.")

        X_scaled = np.array(X_scaled, dtype=float)
        return X_scaled * self.scale_ + self.mean_


class LogisticRegression:
    def __init__(self, learning_rate=0.01, iterations=1000, verbose=False):
        self.learning_rate = learning_rate
        self.iterations = iterations
        self.weights = None
        self.bias = 0
        self.loss_history = []
        self.verbose = verbose

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def loss(self, y_true, y_pred):
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)

        for i in range(self.iterations):
            z = np.dot(X, self.weights) + self.bias
            y_pred = self.sigmoid(z)

            dw = (1 / n_samples) * np.dot(X.T, (y_pred - y))
            db = (1 / n_samples) * np.sum(y_pred - y)

            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

            loss = self.loss(y, y_pred)
            self.loss_history.append(loss)

            if self.verbose:
                print(f"Iteration {i+1}/{self.iterations}: Loss = {loss:.4f}")

    def predict(self, X):
        z = np.dot(X, self.weights) + self.bias
        y_pred = self.sigmoid(z)
        return np.round(y_pred)

scaler = StandardScaler()
X_train_scaler = scaler.fit_transform(X_train)
X_test_scaler = scaler.transform(X_test)

logistic_regression_model = LogisticRegression(learning_rate=0.01, iterations=1000, verbose=False)
logistic_regression_model.fit(X_train_scaler, y_train)

y_pred_test = logistic_regression_model.predict(X_test_scaler)

print("\033[1mLogistic Regression Metric Results:\033[0m")
print(f"Accuracy: {accuracy(y_test, y_pred_test)*100:.2f}%")
print(f"Precision: {precision(y_test, y_pred_test)*100:.2f}%")
print(f"Recall: {recall(y_test, y_pred_test)*100:.2f}%")
print(f"F1 Score: {f1_score(y_test, y_pred_test)*100:.2f}%")


Logistic Regression Metric Results:
Accuracy: 96.81%
Precision: 94.28%
Recall: 94.59%
F1 Score: 94.44%


## Part 5: Multinomial Naive Bayes Implementation

We implement the Multinomial Naive Bayes algorithm from scratch. This variant is specifically chosen over Gaussian Naive Bayes because it is natively suited for discrete frequency counts (like our Bag of Words features), whereas the Gaussian model assumes continuous normally-distributed data.

In [6]:
class MultinomialNaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes = None
        self.class_priors = None
        self.feature_probs = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)

        self.class_priors = np.zeros(n_classes)
        for i, c in enumerate(self.classes):
            self.class_priors[i] = np.sum(y == c) / n_samples

        self.feature_probs = np.zeros((n_classes, n_features))
        for i, c in enumerate(self.classes):
            X_c = X[y == c]
            total_count = np.sum(X_c)
            self.feature_probs[i] = (np.sum(X_c, axis=0) + self.alpha) / (total_count + self.alpha * n_features)

    def predict(self, X):
        n_samples, n_classes = X.shape[0], len(self.classes)
        log_probs = np.zeros((n_samples, n_classes))

        for i in range(n_samples):
            for j in range(n_classes):
                log_prior = np.log(self.class_priors[j])
                log_likelihood = np.sum(X[i] * np.log(self.feature_probs[j]))
                log_probs[i, j] = log_prior + log_likelihood

        return self.classes[np.argmax(log_probs, axis=1)]

model = MultinomialNaiveBayes(alpha=1.0)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("\033[1mMultinomial Naive Bayes Results:\033[0m")
print(f"Accuracy: {accuracy(y_test, y_pred)*100:.2f}%")
print(f"Precision: {precision(y_test, y_pred)*100:.2f}%")
print(f"Recall: {recall(y_test, y_pred)*100:.2f}%")
print(f"F1 Score: {f1_score(y_test, y_pred)*100:.2f}%")


Multinomial Naive Bayes Results:
Accuracy: 95.36%
Precision: 88.75%
Recall: 95.95%
F1 Score: 92.21%
